# MRoPE

**Multi-Resolution Rotary Position Embedding in Qwen3.6-35B-A3B**

This notebook mirrors [the companion mRoPE article section by section](https://huggingface.co/blog/EXDai/mrope). Run the cells
as you read — every output comes from the actual Qwen3.6-35B-A3B model.


---

## Setup

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer
from safetensors import safe_open
import os

print(f"PyTorch {torch.__version__}")

In [ ]:
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3.6-35B-A3B", trust_remote_code=True)
print(f"Vocab: {tok.vocab_size:,}")

# Load embedding weights directly from safetensors (~1GB, first shard only)
snapshot = os.path.expanduser(
    "~/.cache/huggingface/hub/models--Qwen--Qwen3.6-35B-A3B/snapshots/995ad96eacd98c81ed38be0c5b274b04031597b0"
)
with safe_open(os.path.join(snapshot, "model-00001-of-00026.safetensors"), framework="pt") as f:
    embed_weight = f.get_tensor("model.language_model.embed_tokens.weight").float()

print(f"Embedding matrix: {list(embed_weight.shape)}  ({embed_weight.shape[0] * embed_weight.shape[1]:,} params)")


In [ ]:
# mRoPE configuration (from Qwen3.6 config.json — see Ep09)
HEAD_DIM    = 256
HIDDEN_SIZE = 2048
NUM_HEADS   = HIDDEN_SIZE // HEAD_DIM      # 8 pseudo-heads (2048 ÷ 256)
ROPE_THETA  = 10_000_000
ROTARY_DIM  = int(HEAD_DIM * 0.25)         # 64 rotary dims = 32 freq pairs

print(f"Pseudo-heads:   {NUM_HEADS}  (2048 ÷ 256)")
print(f"Rotary dims:    {ROTARY_DIM}  (25% of head_dim)")
print(f"Position-free:  {HEAD_DIM - ROTARY_DIM} dims per head (75%)")
print(f"RoPE theta:     {ROPE_THETA:,}")

In [ ]:
# Build the frequency ladder and the RoPE rotation function
# (Used by all subsequent sections)

def build_freqs(rotary_dim, theta):
    i = torch.arange(0, rotary_dim // 2, dtype=torch.float32)
    return 1.0 / (theta ** (2 * i / rotary_dim))

freqs = build_freqs(ROTARY_DIM, ROPE_THETA)

def apply_rope(x_heads, positions, freqs):
    """x_heads: [batch, seq_len, num_heads, head_dim]"""
    angles = torch.outer(positions.float(), freqs)
    cos, sin = torch.cos(angles), torch.sin(angles)
    rotary_dim = len(freqs) * 2
    x_rot  = x_heads[..., :rotary_dim]
    x_pass = x_heads[..., rotary_dim:]
    x_rot_pairs = x_rot.reshape(*x_rot.shape[:-1], -1, 2)
    cos = cos[None, :, None, :]
    sin = sin[None, :, None, :]
    out = torch.zeros_like(x_rot_pairs)
    out[..., 0] = x_rot_pairs[..., 0] * cos - x_rot_pairs[..., 1] * sin
    out[..., 1] = x_rot_pairs[..., 0] * sin + x_rot_pairs[..., 1] * cos
    return torch.cat([out.reshape(*x_rot.shape), x_pass], dim=-1)

print(f"Frequency pairs: {len(freqs)}")
print(f"apply_rope ready.")

### 1.1 Absolute vs. Relative Position

The naive approach: attach each token to its absolute index.

$$x'_t = x_t + p_t \quad\text{where}\quad p_t = f(t)$$

Now consider what happens when we shift text by inserting whitespace:

```
        the dog barked       ← positions (8, 9, 10)
the dog barked               ← positions (0, 1, 2)
```

The absolute positions are completely different: $(8,9,10)$ vs $(0,1,2)$. But the
**meaning** of the sequence — the relationship between tokens — is identical. An
absolute position encoding forces the model to learn "`the` at position 8
followed by `dog` at position 9" as completely separate from
"`the` at position 0 followed by `dog` at position 1." This is
wasteful: the model re-learns the same linguistic pattern at every possible
coordinate.

What matters for meaning is **relative position** — the offset between tokens — not
where the sequence happens to sit in the document.

### 1.2 What We Want

Return to the whitespace example:

```
        the dog barked       ← positions (8, 9, 10)
the dog barked               ← positions (0, 1, 2)
```

In both versions, `the` and `dog` are adjacent — $q-p = 1$ in both cases. If
position encoding is doing its job, the dot product between these two tokens
(which drives attention) should be the same regardless of whether the pair sits
at $(0,1)$ or $(8,9)$:

$$\langle\,\texttt{the}_R,\; \texttt{dog}_R\,\rangle_{(0,1)} \;=\; \langle\,\texttt{the}_R,\; \texttt{dog}_R\,\rangle_{(8,9)}$$

More generally, a good position encoding should satisfy:

$$\langle\,R_p(a),\; R_q(b)\,\rangle = f(a, b,\; q-p)$$

The dot product between two rotated tokens should depend on:

1. **Who the tokens are** — $a$ and $b$, their semantic embeddings
2. **How far apart they are** — the signed relative offset $q-p$
3. **Nothing else** — not on $p$ or $q$ individually

This is **translation invariance** for relative position. Whether a pair of tokens
appears at $(0, 1)$ or $(1000, 1001)$, the attention score between them is the same.

Among the family of transformations that satisfy this property, **rotation** is the
natural choice. It preserves vector norms (values stay bounded) and has the group
property $R_p^T R_q = R_{q-p}$, which is exactly what delivers translation invariance.

### 1.3 Rotation by Example

Let's make this concrete. Start with a simple vector in 2D:

$$a = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$$

A 2D rotation matrix rotates a vector counterclockwise by angle $\theta$:

$$R_\theta = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

Multiply them:

$$R_\theta \, a = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix} \begin{bmatrix} 1 \\ 0 \end{bmatrix} = \begin{bmatrix} \cos\theta \\ \sin\theta \end{bmatrix}$$

Let's evaluate at specific angles:

$$\theta = 0: \quad \cos(0) = 1,\; \sin(0) = 0 \quad\Rightarrow\quad R_0 a = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$$

$$\theta = \tfrac{\pi}{2}: \quad \cos(\tfrac{\pi}{2}) = 0,\; \sin(\tfrac{\pi}{2}) = 1 \quad\Rightarrow\quad R_{\pi/2} a = \begin{bmatrix} 0 \\ 1 \end{bmatrix}$$

$$\theta = \pi: \quad \cos(\pi) = -1,\; \sin(\pi) = 0 \quad\Rightarrow\quad R_\pi a = \begin{bmatrix} -1 \\ 0 \end{bmatrix}$$

$$\theta = \tfrac{3\pi}{2}: \quad \cos(\tfrac{3\pi}{2}) = 0,\; \sin(\tfrac{3\pi}{2}) = -1 \quad\Rightarrow\quad R_{3\pi/2} a = \begin{bmatrix} 0 \\ -1 \end{bmatrix}$$

$$\theta = 2\pi: \quad \cos(2\pi) = 1,\; \sin(2\pi) = 0 \quad\Rightarrow\quad R_{2\pi} a = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$$

The vector traces the unit circle: right → up → left → down → back to right.
Rotation is an **isometry**: it moves the vector without stretching or shrinking —
values stay bounded no matter how many times the vector is rotated.

In the plane:

```
                  ↑ y
                  │
           ● [0,1]  θ=π/2
                  │
                  │
     ─────────────┼─────────────→ x
     [-1,0] ●     │     ● [1,0]
       θ=π        │      θ=0
                  │
           ● [0,-1]  θ=3π/2
                  │
```

The single vector $a = [1,0]$ spins through these four positions as $\theta$
increases. All four have the same length (1) — rotation moves the vector
around the circle without stretching, so values stay bounded even after
being rotated repeatedly across dozens of transformer layers.

### 1.4 Translation Invariance, Demonstrated


### 1.5 From Toy to Real: What Changes in Actual RoPE

The toy example uses a **single frequency** $\omega = \pi/2$ on a **single 2D vector**.
Now let's build up to what Qwen3.6 actually does, step by step.

**Step 1 — 2048 dimensions.** A token embedding is a vector in $R^{2048}$,
not $R^2$. We split it into smaller subspaces.

**Step 2 — Split into 256-dimensional subspaces.** $2048 \div 256 = 8$. Each 256-D
chunk is a *pseudo-head*. We'll apply the rotation independently within each.

**Step 3 — Within each head, only 64 of the 256 dims rotate.** The remaining 192
are *position-free*. The 64 rotary dimensions pair up to form **32 frequency pairs**.
Each pair rotates like our toy 2D vector, but at its own frequency.

**Step 4 — A different frequency for each pair.** 32 frequencies, from fast
(complete rotation every few positions) to glacial (barely moves). Together they
form a **frequency ladder**.

**Step 5 — 3D multi-resolution.** Qwen3.6 splits the 32 pairs across three spatial
dimensions: 11 temporal (T), 11 height (H), 10 width (W), interleaved THWTHW...

We'll build each of these steps in the cells ahead.

---

## 2. Step 1 & 2 — Loading Real Embeddings

Let's pull out the vector for the token `␣dog` (token ID 5388) and split it
into 8 pseudo-heads.

In [ ]:
dog_id = tok.encode(" dog")[0]   # 5388 — mid-sentence token
dog_emb = embed_weight[dog_id]    # shape: [2048]

print(f"Token ID: {dog_id}")
print(f"Embedding shape: {list(dog_emb.shape)}")
print(f"First 6 values: {[f'{v:.4f}' for v in dog_emb[:6].tolist()]}")

In [ ]:
dog_heads = dog_emb.reshape(NUM_HEADS, HEAD_DIM)  # [8, 256]

print(f"Shape after reshape: {list(dog_heads.shape)}")
print(f"Head 0, first 6 dims: {[f'{v:.4f}' for v in dog_heads[0, :6].tolist()]}")
print(f"Head 1, first 6 dims: {[f'{v:.4f}' for v in dog_heads[1, :6].tolist()]}")
print()
print("Each pseudo-head is an independent 256-D vector.")
print("The rotation will be applied identically within each head.")

---

## 3. Step 3 — Rotary vs. Position-Free Dimensions

Within each 256-D head, only the first 64 dimensions participate in the rotation.
The remaining 192 stay exactly as they are, regardless of position.

In [ ]:
rotary_part = dog_heads[:, :ROTARY_DIM]          # [8, 64]  — these will rotate
free_part   = dog_heads[:, ROTARY_DIM:]           # [8, 192] — these stay frozen

print(f"Rotary part shape:      {list(rotary_part.shape)}  ({NUM_HEADS} heads × {ROTARY_DIM} rotary dims)")
print(f"Position-free part:     {list(free_part.shape)}  ({NUM_HEADS} heads × {HEAD_DIM - ROTARY_DIM} frozen dims)")
print()
print(f"Rotary, head 0, dims 0–3:  {[f'{v:.4f}' for v in rotary_part[0, :4].tolist()]}")
print(f"Free,   head 0, dims 0–3:  {[f'{v:.4f}' for v in free_part[0, :4].tolist()]}")
print()
print("The 64 rotary dims pair up into 32 frequency pairs:")
print("  pair 0: (d0, d1), pair 1: (d2, d3), ..., pair 31: (d62, d63)")
print("Each pair rotates like our toy 2D vector, at its own frequency.")

---

## 4. Step 4 — The Frequency Ladder

Each of the 32 pairs gets its own frequency $\omega_i$:

$$\omega_i = \frac{1}{\theta^{\,2i / d}}, \qquad \theta = 10\,000\,000,\; d = 64$$

In the cell below, we've already built `freqs` in Setup. Let's inspect them.

In [ ]:
wavelengths = 2 * torch.pi / freqs

print(f"θ = {ROPE_THETA:,}")
print()
for idx in [0, 10, 20, 31]:
    wl = wavelengths[idx].item()
    if wl < 1000:
        wl_str = f"{wl:.1f} positions"
    elif wl < 1_000_000:
        wl_str = f"{wl:,.0f} positions"
    else:
        wl_str = f"{wl:.1e} positions"
    print(f"Pair {idx:>2}: ω = {freqs[idx].item():>16.12f},  λ = {wl_str}")

print()
print("The fastest pair completes a rotation every ~6 tokens.")
print("The slowest pair has barely moved after 262,144 tokens (full context).")

---

## 5. Putting It Together — Rotating a Real Embedding

We now have all the pieces. Let's rotate the `␣dog` embedding at position 0
(no rotation) and position 5.

In [ ]:
dog_h = dog_heads.unsqueeze(0).unsqueeze(0)  # [1, 1, 8, 256]
rot0 = apply_rope(dog_h, torch.tensor([0]), freqs).squeeze()
rot5 = apply_rope(dog_h, torch.tensor([5]), freqs).squeeze()

print("Head 0, rotary dims 0–3:")
print(f"  at pos 0: {[f'{v:.4f}' for v in rot0[0, :4].tolist()]}")
print(f"  at pos 5: {[f'{v:.4f}' for v in rot5[0, :4].tolist()]}  ← rotated!")
print()
print("Head 0, position-free dims 64–67:")
print(f"  at pos 0: {[f'{v:.4f}' for v in rot0[0, 64:68].tolist()]}")
print(f"  at pos 5: {[f'{v:.4f}' for v in rot5[0, 64:68].tolist()]}  ← unchanged!")
print()
print("The rotary dimensions shift with position.")
print("The position-free dimensions stay frozen.")
print("This is the 25/75 split in action.")

---

## 6. Translation Invariance — The Payoff

Finally, let's verify that the same token pair at the same relative offset produces
the same dot product, regardless of where it sits in the sentence.

```
and the cat and the dog and the cat
     ^^^                          ^^^
  (the,cat)                    (the,cat)
  positions (1,2)              positions (7,8)
```

In [ ]:
sentence = "and the cat and the dog and the cat"
token_ids = torch.tensor(tok.encode(sentence))
tokens = tok.convert_ids_to_tokens(token_ids)

print("Sentence tokenized:")
for i, (tid, tok_str) in enumerate(zip(token_ids, tokens)):
    print(f"  pos {i:>2}: id={tid:>6}  '{tok_str.replace('Ġ', '␣')}'")

def pair_dot(p, q):
    positions = torch.tensor([p, q], dtype=torch.long)
    emb = embed_weight[token_ids[[p, q]]]  # [2, 2048]
    emb_h = emb.reshape(2, NUM_HEADS, HEAD_DIM).unsqueeze(0)
    rotated = apply_rope(emb_h, positions, freqs).reshape(2, -1)
    return torch.dot(rotated[0], rotated[1]).item()

print()
print(f"{'Pair':<20} {'positions':>10} {'q-p':>5} {'dot product':>14}")
print("-" * 52)

early = pair_dot(1, 2)
late  = pair_dot(7, 8)
dog   = pair_dot(4, 5)

print(f"{' (the,cat) early':<20} {'(1,2)':>10} {'1':>5} {early:>14.6f}")
print(f"{' (the,cat) late ':<20} {'(7,8)':>10} {'1':>5} {late:>14.6f}")
print(f"{' (the,dog)      ':<20} {'(4,5)':>10} {'1':>5} {dog:>14.6f}")
print()
print(f"(the,cat) early vs late diff: {early - late:.2e}  ← zero!")
print(f"(the,cat) vs (the,dog) diff:  {abs(early - dog):.6f}  ← different tokens")

---

## Summary

Same token pair, same $q-p$, different absolute positions → identical dot product
to 10 decimal places. Different token pair at the same offset → different dot
product. This is exactly the property we asked for in §1.2: the attention score
between two tokens depends on **who they are** and **how far apart they are** —
not on where they sit in the sequence.

| Concept | What we built |
|---------|--------------|
| **Rotation** | Each token's embedding spins through space as position increases |
| **Frequency ladder** | 32 frequencies from fast (λ=6) to glacial (λ=38M) |
| **25/75 split** | 64 rotary dims encode position; 192 frozen dims carry semantics |
| **Translation invariance** | Dot product depends on $q-p$, not on $(p,q)$ |

**Next:** Building the full mRoPE — 3D sections [11,11,10], interleaving,
and what changes for multimodal input.